In [20]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub


In [21]:
from langchain.agents import create_react_agent, AgentExecutor

In [22]:
# ================================
# LOAD ENV VARIABLES
# ================================

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [23]:
search_tool = TavilySearchResults(max_results =2)

In [25]:
@ tool
def get_weather_data(city: str)-> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']} degree C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [9]:
result = search_tool.invoke("Give me the latest news on Trump")

In [26]:
# =============================
# LLM
# =============================

llm = ChatOpenAI(
    model = "gpt-3.5-turbo",
    temperature = 0,
    api_key = OPENAI_API_KEY
)

In [13]:
response = llm.invoke("What year is it?")
response

AIMessage(content='It is currently 2022.', response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 12, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-e776427f-44a6-4c21-9284-7d689ece21d9-0')

In [27]:
# ==============================
# PROMPT
# ==============================

prompt = hub.pull("hwchase17/react")

c:\Users\91810\anaconda\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [28]:
# ===========================
# Tools
# ===========================

tools = [search_tool, get_weather_data]

In [29]:
# ==================================
# CREATE AGENT
# ==================================

agent = create_react_agent(
    llm = llm,
    tools = tools,
    prompt = prompt
)

In [30]:
# ==================================
# AGENT EXECUTOR
# ==================================

agent_executor = AgentExecutor(
    agent =agent,
    tools = tools,
    verbose = True
)

In [31]:
# ====================================
# RUN
# ====================================

response = agent_executor.invoke({
    "input": (
        " Tell me the capital of Australia and its current weather?"
    )
})



> Entering new AgentExecutor chain...
I should use the search engine to find the capital of Australia and then use the weather data tool to get the current weather information.
Action: tavily_search_results_json
Action Input: "capital of Australia"[{'url': 'https://youtooproject.com/en/blog/australia-en/capital-of-australia-canberra', 'content': '## The origin of Australia’s capital\n\nLet’s give you a little background. It is 1901. The Commonwealth Parliament has just been established. Even though the grand opening ceremony was to be held in Melbourne, no city had yet been designated as the capital of Australia.\n\nObviously, the two major cities of the country, Sydney and Melbourne, started discussing who should be the central Australian hub. However, neither was really an appealing choice.\n\nSince both cities were located on the coast, they were vulnerable to a possible naval attack. Faced with widespread chaos, a radical decision was made to cut to the chase: they would build a 

In [32]:
print(response["output"])

The capital of Australia is Canberra and the current weather there is overcast with a temperature of 13 degrees Celsius and 56% humidity.
